In [0]:
schemalocation = "/Volumes/ev_spark/myvol/checkpoint/schema/alt_fuel_stns/"
sourcepath = "/Volumes/ev_spark/myvol/landing/alt_fuel_stns/"
chkptpath = "/Volumes/ev_spark/myvol/checkpoint/chkpt/alt_fuel_stns/"

## Method to read Alt_Fuel_Station data into a data frame

In [0]:
def readRawFuelStn():
    from pyspark.sql.functions import current_timestamp
    rawFuelStn_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schemalocation)
    .option("cloudFiles.inferSchema", "true")
    .load(sourcepath)
    .withColumn("extraction_date", current_timestamp())
    )
    print("Reading success!")
    print("***********************")
    return rawFuelStn_df



## Method to write Alt_Fuel_Station data into a delta table

In [0]:
def writeBronzeFuelStn(read_df):

    from pyspark.sql.functions import col
    import re

    # Sanitize column names by replacing invalid characters with underscores
    clean_df = read_df
    for original_col in clean_df.columns:
        clean_col = re.sub(r'[ ,;{}()\n\t=]', '_', original_col)
        if clean_col != original_col:
            clean_df = clean_df.withColumnRenamed(original_col, clean_col)

    (clean_df.writeStream
    .format("delta")
    .option("checkpointLocation", chkptpath)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("ev_spark.bronze.alt_fuel_stns"))
    print("writing to bronze successful!")

## Calling read and write methods for Alt_Fuel_Stations data

In [0]:
readRawFuelStn_df = readRawFuelStn()
writeBronzeFuelStn(readRawFuelStn_df)

In [0]:
%sql
select * from ev_spark.bronze.alt_fuel_stns limit 10;

In [0]:
readRawFuelStn_df.printSchema()